In [1]:
import os, io, base64, numpy as np, requests, torch, torch.nn as nn
from PIL import Image

BASE_URL = os.getenv("BASE_URL", "http://154.57.164.70:36262")  # <-- export BASE_URL=http://instance_ip:port
MNIST_MEAN, MNIST_STD = 0.1307, 0.3081

# ---------- codecs ----------
def x01_from_b64_png(b64):
    raw = base64.b64decode(b64)
    img = Image.open(io.BytesIO(raw)).convert("L")
    if img.size != (28, 28):
        raise ValueError("Expected 28x28 PNG")
    return np.clip(np.asarray(img, dtype=np.float32) / 255.0, 0.0, 1.0)

def b64_png_from_x01(x2d):
    x255 = np.clip((x2d * 255.0).round(), 0, 255).astype(np.uint8)
    buf = io.BytesIO()
    Image.fromarray(x255, mode="L").save(buf, format="PNG", optimize=True)
    return base64.b64encode(buf.getvalue()).decode("ascii")

def linf(a, b):
    return float(np.max(np.abs(a - b)))

# ---------- model ----------
class SimpleClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)
    def forward(self, x01):
        x = (x01 - MNIST_MEAN) / MNIST_STD
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)
        return torch.log_softmax(x, dim=1)

# ---------- fetch challenge + weights ----------
ch  = requests.get(f"{BASE_URL}/challenge", timeout=10).json()
x   = x01_from_b64_png(ch["image_b64"])       # (28,28) in [0,1]
lab = int(ch["label"])
eps = float(ch["epsilon"])
print("baseline label:", lab, "epsilon:", eps)

open("fgsm_weights.pth", "wb").write(requests.get(f"{BASE_URL}/weights", timeout=10).content)
model = SimpleClassifier().eval()
model.load_state_dict(torch.load("fgsm_weights.pth", map_location="cpu"))

# sanity: local pred must match server
xt = torch.from_numpy(x[None, None, ...]).float()
print("local pred:", int(model(xt).argmax(1)), "(should match server)")

# ---------- FGSM ----------
def fgsm(x_np, epsilon):
    xt = torch.from_numpy(x_np[None, None, ...]).float().requires_grad_(True)
    out = model(xt)                                  # log-softmax
    loss = nn.functional.nll_loss(out, torch.tensor([lab]))
    model.zero_grad(); loss.backward()
    grad_sign = xt.grad.sign().squeeze().numpy()     # (28,28)
    adv = np.clip(x_np + epsilon * grad_sign, 0.0, 1.0)
    return adv.astype(np.float32)

# Try the full budget, then shrink until the PNG-rounded L_inf is within eps
# (rounding can nudge a pixel over the bound, so verify the decoded image).
for scale in [1.0, 0.98, 0.96, 0.94, 0.92, 0.9, 0.85, 0.8]:
    adv = fgsm(x, eps * scale)
    b64 = b64_png_from_x01(adv)
    adv_rt = x01_from_b64_png(b64)                   # what the server will actually see
    d = linf(adv_rt, x)
    pred = int(model(torch.from_numpy(adv_rt[None, None, ...]).float()).argmax(1))
    print(f"scale={scale:.2f}  linf(after PNG)={d:.4f}  local_pred={pred}")
    if d <= eps and pred != lab:
        print("  -> candidate OK, submitting")
        break

# ---------- submit ----------
r = requests.post(f"{BASE_URL}/submit", json={"image_b64": b64}, timeout=10)
print(r.status_code, r.text)

baseline label: 1 epsilon: 0.25
local pred: 1 (should match server)
scale=1.00  linf(after PNG)=0.2510  local_pred=8
scale=0.98  linf(after PNG)=0.2431  local_pred=8
  -> candidate OK, submitting
200 {"ok":true,"flag":"[REDACTED]","pred":8,"linf":0.24313727021217346}
